<a href="https://colab.research.google.com/github/OlhaZahrebelna/certflow-rag-assistant/blob/main/src/rag/02_generation_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generation Evaluation for the End-to-End RAG Baseline

This notebook evaluates the quality of final RAG answers produced by the selected dense retrieval pipeline.

The evaluation focuses on four generation-level criteria:

- **Correctness** — whether the answer matches the reference answer.
- **Faithfulness** — whether substantive claims are supported by retrieved context.
- **Relevance** — whether the answer directly addresses the user question.
- **Source grounding** — whether cited sources are supported by the retrieved context.

The retrieval configuration is kept fixed so that this notebook evaluates generation quality rather than retrieval alternatives.

## 1. Setup

In [1]:
!pip install -q openai sentence-transformers faiss-cpu

In [2]:
import json
import numpy as np
import faiss
import re

from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import userdata

## 2. Load the repository and API key

In [3]:
repo_path = Path("/content/certflow-rag-assistant")

if not repo_path.exists():
    !git clone https://github.com/OlhaZahrebelna/certflow-rag-assistant.git
else:
    print("Repository already exists.")

Repository already exists.


In [4]:
%cd /content/certflow-rag-assistant

/content/certflow-rag-assistant


In [5]:
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

client = OpenAI(
    api_key=OPENAI_API_KEY
)

## 3. Load processed chunks

In [6]:
chunks_path = Path("data/raw/processed/chunks.json")

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [chunk["content"] for chunk in chunks]

print(f"Loaded chunks: {len(chunks)}")

Loaded chunks: 81


## 4. Rebuild the selected dense retriever

The retrieval stage uses the best-performing baseline from the retrieval experiments:

`multi-qa-MiniLM-L6-cos-v1` + normalized embeddings + FAISS inner-product search.

In [7]:
embedding_model = SentenceTransformer(
    "multi-qa-MiniLM-L6-cos-v1"
)

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"Embedding shape: {embeddings.shape}")
print(f"Vectors indexed: {index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape: (81, 384)
Vectors indexed: 81


In [8]:
def retrieve(query, k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    scores, indices = index.search(
        query_embedding,
        k=k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "chunk_id": chunk["chunk_id"],
            "document": chunk["metadata"]["title"],
            "section": chunk["metadata"]["section"],
            "content": chunk["content"],
        })

    return results

## 5. Build context and generate grounded answers

In [9]:
def build_context(retrieved_chunks):
    parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        parts.append(
            f"""Source {i}
Document: {chunk['document']}
Section: {chunk['section']}
Content:
{chunk['content']}"""
        )

    return "\n\n---\n\n".join(parts)

In [10]:
SYSTEM_PROMPT = """
You are CertFlow, an internal Account Data Certification assistant.

Answer the user's question using only the provided context.

Rules:
- Do not use outside knowledge.
- Do not invent policies, procedures, or requirements.
- If the context is insufficient, say that the available documentation is insufficient.
- Answer the question directly and concisely.
- Include a short Sources section with the supporting document and section names.
"""


In [11]:
GENERATION_MODEL = "gpt-5.6-luna"

def generate_answer(query, k=5):
    retrieved_chunks = retrieve(query, k=k)
    context = build_context(retrieved_chunks)

    user_prompt = f"""
Question:
{query}

Context:
{context}

Answer the question based only on the context above.
"""

    response = client.responses.create(
        model=GENERATION_MODEL,
        instructions=SYSTEM_PROMPT,
        input=user_prompt
    )

    return {
        "query": query,
        "answer": response.output_text,
        "retrieved_chunks": retrieved_chunks
    }

## 6. Generation evaluation dataset

Reference answers are intentionally concise. They are used as evaluation anchors rather than text that the generated answer must reproduce verbatim.

In [12]:
generation_eval_dataset = [
    {
        "query": "What evidence is required before an account can be certified?",
        "reference_answer": (
            "Evidence must be stored and traceable. The certification record should include "
            "the source level, exact source or vendor name, URL or internal reference, "
            "access or publication date, supported fields, a paraphrased evidence excerpt, "
            "and the analyst conclusion."
        ),
        "expected_sources": [
            "Source Hierarchy and Evidence Standard",
            "End-to-End Account Certification Workflow"
        ]
    },

    {
        "query": "How should potential duplicate accounts be handled?",
        "reference_answer": (
            "Potential duplicates must be screened before certification. If a duplicate is "
            "confirmed, the appropriate duplicate outcome should be applied, such as merge, "
            "escalation, or retaining separate records when they represent different entities."
        ),
        "expected_sources": [
            "Address Certification and Duplicate Prevention",
            "End-to-End Account Certification Workflow"
        ]
    },

    {
        "query": "When should a certification case be escalated?",
        "reference_answer": (
            "A case should be escalated according to the defined escalation levels when it "
            "cannot be resolved through the normal certification or QA process and requires "
            "additional governance or higher-level review."
        ),
        "expected_sources": [
            "Quality Review, Exceptions, and Escalations"
        ]
    },

    {
        "query": "What is the responsibility of the Quality Assurance Reviewer?",
        "reference_answer": (
            "The Quality Assurance Reviewer independently checks certification quality, "
            "verifies compliance with required controls, reviews evidence and decisions, "
            "and records the QA outcome."
        ),
        "expected_sources": [
            "Roles and Responsibilities"
        ]
    },

    {
        "query": "What validation rules apply to account fields?",
        "reference_answer": (
            "Account fields are validated according to the field catalog and their defined "
            "rules, including required values, formats, permitted values, evidence expectations, "
            "and field-specific certification requirements."
        ),
        "expected_sources": [
            "Account Fields and Validation Rules"
        ]
    },

    {
        "query": "What should an analyst do when two sources conflict?",
        "reference_answer": (
            "When sources conflict, the analyst should compare source authority and freshness, "
            "investigate the discrepancy, document the evidence and reasoning, and avoid "
            "certification until the conflict is resolved or appropriately escalated."
        ),
        "expected_sources": [
            "Source Hierarchy and Evidence Standard"
        ]
    },

    {
        "query": "Which sources are preferred when validating account information?",
        "reference_answer": (
            "Primary sources should be preferred where available. Secondary sources may be "
            "used when appropriate and should be assessed according to the source hierarchy "
            "and evidence standards."
        ),
        "expected_sources": [
            "Source Hierarchy and Evidence Standard"
        ]
    },

    {
        "query": "What information must be recorded when an account certification change is made?",
        "reference_answer": (
            "The change record should capture the relevant audit information, including what "
            "changed, the reason for the change, supporting evidence, the responsible analyst, "
            "and other required change-record details."
        ),
        "expected_sources": [
            "Request Types and Change Management",
            "End-to-End Account Certification Workflow"
        ]
    },

    {
        "query": "What information is required when submitting a new certification request?",
        "reference_answer": (
            "A certification request must include enough information to identify the account "
            "and define the requested work. If a usable Account ID is unavailable, sufficient "
            "identifying information such as legal name, country, website, or address must be provided."
        ),
        "expected_sources": [
            "Request Types and Change Management",
            "End-to-End Account Certification Workflow"
        ]
    },

    {
        "query": "How should an analyst verify that they are reviewing the correct company entity?",
        "reference_answer": (
            "The analyst should verify the entity using identifying attributes and available "
            "evidence, such as legal name, website or domain, address, external identifiers, "
            "country, and other relevant identity information."
        ),
        "expected_sources": [
            "End-to-End Account Certification Workflow"
        ]
    },

    {
        "query": "What checks are required before an account can be marked as Verified?",
        "reference_answer": (
            "Before an account can be marked as Verified, mandatory fields must be resolved, "
            "supporting evidence must be documented, required changes must be explained, "
            "duplicate screening must be complete, and required QA must have passed."
        ),
        "expected_sources": [
            "End-to-End Account Certification Workflow"
        ]
    },

    {
        "query": "When is QA review mandatory?",
        "reference_answer": (
            "QA review is mandatory when a case meets one or more defined mandatory QA triggers "
            "described in the quality review policy."
        ),
        "expected_sources": [
            "Quality Review, Exceptions, and Escalations"
        ]
    },

    {
        "query": "What should be checked during a QA review?",
        "reference_answer": (
            "QA should verify that the certification decision, evidence, field values, required "
            "controls, audit documentation, and relevant policy requirements have been applied correctly."
        ),
        "expected_sources": [
            "Quality Review, Exceptions, and Escalations"
        ]
    },

    {
        "query": "What happens when a certification case fails QA?",
        "reference_answer": (
            "A failed QA review should result in the case being returned or routed according to "
            "the defined QA outcomes so that defects can be corrected before certification is completed."
        ),
        "expected_sources": [
            "Quality Review, Exceptions, and Escalations"
        ]
    },

    {
        "query": "How should missing address information be handled?",
        "reference_answer": (
            "Missing address information must be handled according to the address certification "
            "rules. The analyst should document the missing information and use the defined process "
            "for cases where required address evidence cannot be obtained."
        ),
        "expected_sources": [
            "Address Certification and Duplicate Prevention"
        ]
    },

    {
        "query": "How should an address be normalized before certification?",
        "reference_answer": (
            "Address data should be standardized according to the documented normalization rules "
            "so that components are represented consistently before certification."
        ),
        "expected_sources": [
            "Address Certification and Duplicate Prevention"
        ]
    },

    {
        "query": "How should an account's Legal Name be validated or updated?",
        "reference_answer": (
            "The Legal Name should be validated against appropriate authoritative evidence and "
            "updated according to the Legal Name field rules, with supporting evidence and change "
            "documentation where required."
        ),
        "expected_sources": [
            "Account Fields and Validation Rules"
        ]
    },

    {
        "query": "How should a website or primary domain be validated?",
        "reference_answer": (
            "The website and primary domain should be validated using reliable evidence and checked "
            "against the field-specific rules to ensure they correspond to the correct business entity."
        ),
        "expected_sources": [
            "Account Fields and Validation Rules"
        ]
    },

    {
        "query": "What should an analyst do if the preferred source is unavailable?",
        "reference_answer": (
            "If the preferred source is unavailable, the analyst should follow the documented "
            "source hierarchy, use an acceptable alternative source where permitted, and record "
            "the evidence and limitations appropriately."
        ),
        "expected_sources": [
            "Source Hierarchy and Evidence Standard"
        ]
    },

    {
        "query": "When should a certified account be reviewed again?",
        "reference_answer": (
            "A certified account should be reviewed again when certification validity requirements "
            "indicate that re-certification is needed, such as after relevant changes, new evidence, "
            "or when the certification is no longer considered current."
        ),
        "expected_sources": [
            "Account Data Certification Overview"
        ]
    }
]

## 7. Generate answers for the evaluation set

In [13]:
generation_results = []

for item in generation_eval_dataset:
    result = generate_answer(item["query"], k=5)

    generation_results.append({
        "query": item["query"],
        "reference_answer": item["reference_answer"],
        "expected_sources": item["expected_sources"],
        "generated_answer": result["answer"],
        "retrieved_chunks": result["retrieved_chunks"]
    })

print(f"Generated answers: {len(generation_results)}")


Generated answers: 20


## 8. LLM-as-a-Judge evaluation

In [14]:
def format_retrieved_context(retrieved_chunks):
    return build_context(retrieved_chunks)

In [15]:
JUDGE_PROMPT = """
You are evaluating a Retrieval-Augmented Generation system.

Evaluate the generated answer using four criteria.

1. Correctness
0 = materially incorrect
1 = partially correct
2 = correct according to the reference answer

2. Faithfulness
0 = contains substantive claims unsupported by the retrieved context
1 = mostly supported, with minor unsupported or overstated details
2 = all substantive claims are supported by the retrieved context

3. Relevance
0 = does not answer the question
1 = answers the question but includes unnecessary or distracting information
2 = directly and concisely answers the question

4. Source grounding
0 = cited sources are missing, incorrect, or unsupported
1 = sources are partially correct
2 = cited sources match the supporting retrieved context and expected sources

Return ONLY valid JSON in this exact format:
{
  "correctness": 0,
  "faithfulness": 0,
  "relevance": 0,
  "source_grounding": 0,
  "reason": "short explanation"
}
"""


In [16]:
JUDGE_MODEL = "gpt-5.6-luna"

def parse_judge_json(text):
    text = text.strip()

    # Handle an occasional Markdown code fence around otherwise valid JSON.
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    return json.loads(text)


def evaluate_answer(
    query,
    generated_answer,
    reference_answer,
    expected_sources,
    retrieved_chunks
):
    context = format_retrieved_context(retrieved_chunks)

    evaluation_input = f"""
Question:
{query}

Reference answer:
{reference_answer}

Expected sources:
{json.dumps(expected_sources, ensure_ascii=False)}

Retrieved context:
{context}

Generated answer:
{generated_answer}
"""

    response = client.responses.create(
        model=JUDGE_MODEL,
        instructions=JUDGE_PROMPT,
        input=evaluation_input
    )

    return parse_judge_json(response.output_text)


## 9. Run generation evaluation

In [17]:
evaluated_results = []

for item in generation_results:
    evaluation = evaluate_answer(
        query=item["query"],
        generated_answer=item["generated_answer"],
        reference_answer=item["reference_answer"],
        expected_sources=item["expected_sources"],
        retrieved_chunks=item["retrieved_chunks"]
    )

    evaluated_results.append({
        **item,
        **evaluation
    })

print(f"Evaluated answers: {len(evaluated_results)}")

Evaluated answers: 20


## 10. Inspect individual results

In [18]:
for result in evaluated_results:
    print("\n" + "=" * 80)

    print("QUERY:")
    print(result["query"])

    print("\nANSWER:")
    print(result["generated_answer"])

    print("\nSCORES:")
    print("Correctness:", result["correctness"])
    print("Faithfulness:", result["faithfulness"])
    print("Relevance:", result["relevance"])
    print("Source grounding:", result["source_grounding"])

    print("\nReason:")
    print(result["reason"])


QUERY:
What evidence is required before an account can be certified?

ANSWER:
Before an account can be certified, the analyst must have:

- Evidence resolving all mandatory fields in scope.
- Recorded evidence including:
  - Source level (Primary or Secondary)
  - Exact source or vendor name
  - URL or internal source reference
  - Access or publication date
  - Fields supported
  - A paraphrased evidence excerpt
  - The analyst’s conclusion
- Reasons documented for any changes.
- Completed duplicate screening.
- Passed required QA.

The available documentation does not specify which particular source types or field-specific evidence are mandatory.

**Sources**
- *End-to-End Account Certification Workflow* — Section 9, “Certification decision”
- *Source Hierarchy and Evidence Standard* — Section 6, “Evidence recording”

SCORES:
Correctness: 2
Faithfulness: 2
Relevance: 2
Source grounding: 2

Reason:
The answer accurately lists the required traceable evidence elements and the additiona

## 11. Aggregate generation metrics

In [20]:
metrics = [
    "correctness",
    "faithfulness",
    "relevance",
    "source_grounding"
]

print("\nGENERATION EVALUATION — 20 QUESTIONS")
print("=" * 50)

for metric in metrics:
    scores = [
        result[metric]
        for result in evaluated_results
    ]

    average = np.mean(scores)

    print(
        f"{metric:18s}: "
        f"{average:.2f} / 2.00"
    )


GENERATION EVALUATION — 20 QUESTIONS
correctness       : 1.80 / 2.00
faithfulness      : 1.95 / 2.00
relevance         : 1.95 / 2.00
source_grounding  : 1.85 / 2.00


In [21]:
perfect_answers = sum(
    result["correctness"] == 2
    and result["faithfulness"] == 2
    and result["relevance"] == 2
    and result["source_grounding"] == 2
    for result in evaluated_results
)

print(
    f"\nPerfect answers: "
    f"{perfect_answers}/{len(evaluated_results)}"
)


Perfect answers: 14/20


In [22]:
for result in evaluated_results:

    if (
        result["correctness"] < 2
        or result["faithfulness"] < 2
        or result["relevance"] < 2
        or result["source_grounding"] < 2
    ):
        print("\n" + "=" * 80)
        print("QUERY:")
        print(result["query"])

        print("\nSCORES:")
        print("Correctness:", result["correctness"])
        print("Faithfulness:", result["faithfulness"])
        print("Relevance:", result["relevance"])
        print(
            "Source grounding:",
            result["source_grounding"]
        )

        print("\nReason:")
        print(result["reason"])


QUERY:
When should a certification case be escalated?

SCORES:
Correctness: 1
Faithfulness: 1
Relevance: 2
Source grounding: 1

Reason:
The answer identifies several supported escalation triggers, including unresolved conflicts, uncertain identity, missing evidence, and required QA or Governance review. However, it does not clearly state the overarching rule that escalation occurs when the normal certification or QA process cannot resolve the case and higher-level governance is required. It also treats any material change or contradictory evidence as escalation, while the context only says these trigger review. The cited sources support the details but do not match the expected source on escalation levels.

QUERY:
What is the responsibility of the Quality Assurance Reviewer?

SCORES:
Correctness: 1
Faithfulness: 2
Relevance: 2
Source grounding: 2

Reason:
The answer accurately describes the independent QA checks and supported handling of corrections, but omits the responsibility to re

## Generation Evaluation Results

The end-to-end RAG baseline was evaluated on 20 representative policy questions using an LLM-as-a-judge framework.

| Metric | Average Score |
|---|---:|
| Correctness | **1.80 / 2.00** |
| Faithfulness | **1.95 / 2.00** |
| Relevance | **1.95 / 2.00** |
| Source Grounding | **1.85 / 2.00** |

**14 of 20 answers (70%) achieved perfect scores across all four criteria.**

The strongest results were observed for faithfulness and relevance, showing that the model generally stays grounded in the retrieved context and answers the user's question directly.

Most remaining errors were caused by one of three factors:

1. the most specific policy section was not retrieved;
2. the generated answer omitted a required detail;
3. the answer cited valid supporting context but not the expected canonical source.

These results indicate that the generation component is reliable when high-quality context is retrieved. Further improvements should focus primarily on retrieval precision, source selection, and reducing unnecessary supporting detail rather than changing the generation model.